# Run 9 — Bare-PCB vs. PCBA Classification: 3-Model Comparative Benchmark

**Goal:** binary image classification — is this a **bare PCB** (no components mounted) or a
**PCBA** (populated / assembled board)? Framed as a candidate **pre-filter stage** ahead of the
defect-detection pipeline (Runs 1-8), so the router can decide whether an incoming image needs
bare-board trace-defect detection or a different (component-level) inspection path.

**Datasets (bare_pcb class):**
- **PKU-PCB** (`Ironbrotherstyle/PCB-DATASET`) — same source as Run 1, all defect-folder images pooled, VOC/annotations ignored.
- **DeepPCB** (`tangsanli5201/DeepPCB`) — same source as Runs 3-8, both template and test-diff images pooled.

**Dataset (pcba class):**
- **Roboflow Universe — `pcbdataset-abppt/pcb-components-rkizn`** — populated-board component-detection
  dataset; we discard the bounding boxes and use the raw images as PCBA-class exemplars.

**Models compared (3):**

| Model | Params | Why |
|---|---|---|
| YOLOv11n-cls | ~1.6M | Same Ultralytics stack already used for detection — drops straight into existing W&B logging |
| EfficientNet-B0 | ~5.3M | Standard transfer-learning baseline, easy to benchmark against literature norms |
| MobileNetV3-Small | ~2.5M | Lightest/fastest — the natural choice if this becomes a pre-filter stage |

YOLOv11s-cls is wired in as an optional 4th row (`INCLUDE_YOLO_S_CLS = True/False` below) since the
brief lists it as an n/s alternative rather than a required 4th model.

**Pipeline:** clone/verify raw dataset repos → build bare_pcb pool → download PCBA pool from Roboflow →
stratified 70/15/15 split → train all models with matched protocol (seed, epochs, img size) → evaluate
on held-out test split → benchmark accuracy, params, model size, CPU/GPU latency → comparison table +
plots → log everything to W&B.

Run top to bottom on a machine with the bare-PCB dataset repos reachable (or already cloned) and a
`ROBOFLOW_API_KEY` set for the PCBA download.

## 1. Environment check

In [ ]:
import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

In [ ]:
import subprocess, sys

def _ensure(pkgs, pip_names=None):
    pip_names = pip_names or pkgs
    missing = []
    for mod, pip_name in zip(pkgs, pip_names):
        try:
            __import__(mod)
        except ImportError:
            missing.append(pip_name)
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

_ensure(
    ["ultralytics", "torchvision", "roboflow", "wandb", "sklearn", "seaborn",
     "pandas", "cv2", "yaml", "thop"],
    ["ultralytics", "torchvision", "roboflow", "wandb", "scikit-learn", "seaborn",
     "pandas", "opencv-python-headless", "pyyaml", "thop"],
)

import ultralytics, torchvision, wandb, cv2
print("ultralytics :", ultralytics.__version__)
print("torchvision :", torchvision.__version__)
print("wandb       :", wandb.__version__)
print("opencv      :", cv2.__version__)

## 2. Configuration

Mirrors the `PROJECT_ROOT` resolution and seed/config conventions used in Runs 1-8.

In [ ]:
from pathlib import Path
import os

_cwd = Path(os.getcwd()).resolve()
ROOT_ENV = os.environ.get("PCB_PROJECT_ROOT")
if ROOT_ENV:
    PROJECT_ROOT = Path(ROOT_ENV).resolve()
elif _cwd.name == "experiments":
    PROJECT_ROOT = _cwd.parent
elif (_cwd / "experiments").exists():
    PROJECT_ROOT = _cwd
else:
    PROJECT_ROOT = _cwd

SEED, IMG_SIZE, BATCH, DEVICE = 42, 224, 32, 0        # DEVICE='cpu' if no GPU
EPOCHS       = 50                                      # classification converges much faster than detection
PATIENCE     = 15
EXP_NAME     = "exp_009_bare_vs_pcba_classification"
CLASS_NAMES  = ["bare_pcb", "pcba"]                    # index 0 / 1

# Raw bare-PCB dataset repos (already used by Runs 1 & 3-8; cloned here if missing)
PKU_DIR      = PROJECT_ROOT / "PCB-DATASET"
DEEPPCB_DIR  = PROJECT_ROOT / "DeepPCB"

# Roboflow PCBA source
ROBOFLOW_WORKSPACE = "pcbdataset-abppt"
ROBOFLOW_PROJECT   = "pcb-components-rkizn"
ROBOFLOW_VERSION   = None      # None = auto-pick latest published version
ROBOFLOW_DIR       = PROJECT_ROOT / "pcba_roboflow"

# Output classification dataset (ImageFolder-style: split/class/*.jpg — works for both
# Ultralytics -cls training and torchvision ImageFolder)
CLS_DATASET_DIR = PROJECT_ROOT / "pcb_vs_pcba_dataset"
SPLIT_RATIOS    = dict(train=0.70, val=0.15, test=0.15)
BALANCE_CLASSES = True         # undersample the majority class to the minority count

RESULTS_DIR = PROJECT_ROOT / "results"; RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR    = PROJECT_ROOT / "runs" / EXP_NAME; RUNS_DIR.mkdir(parents=True, exist_ok=True)

INCLUDE_YOLO_S_CLS = False     # flip to True to add YOLOv11s-cls as a 4th comparison row

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project root      :", PROJECT_ROOT)
print("PKU dir present    :", PKU_DIR.exists())
print("DeepPCB dir present:", DEEPPCB_DIR.exists())
print("Classification data:", CLS_DATASET_DIR)

## 3. Assemble the `bare_pcb` pool

### 3a. PKU-PCB (`Ironbrotherstyle/PCB-DATASET`)
Same clone step as Run 1. We ignore the VOC annotations entirely here — every image under
`images/**` is a bare (unpopulated) PCB, which is all a binary bare-vs-populated classifier needs.

In [ ]:
import subprocess

if not (PKU_DIR / ".git").exists():
    print("Cloning PKU-PCB (Ironbrotherstyle/PCB-DATASET)...")
    r = subprocess.run(["git", "clone", "--depth", "1",
                         "https://github.com/Ironbrotherstyle/PCB-DATASET", str(PKU_DIR)],
                        capture_output=True, text=True)
    print((r.stdout or "")[-400:]); print((r.stderr or "")[-400:])
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed - download the ZIP manually into {PKU_DIR}")
else:
    print("PKU-PCB already present at", PKU_DIR)

assert (PKU_DIR / "images").exists(), f"Expected {PKU_DIR/'images'} - dataset structure changed?"
pku_images = sorted(p for p in (PKU_DIR / "images").rglob("*")
                     if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
print(f"PKU-PCB bare-board images found: {len(pku_images)}")

### 3b. DeepPCB (`tangsanli5201/DeepPCB`)
Same clone step as Runs 3-8. Every image under `PCBData/**` (both the clean *template* and the
*test*-diff shots) is a bare board — labels are again irrelevant for this task.

In [ ]:
if not (DEEPPCB_DIR / ".git").exists() and not DEEPPCB_DIR.exists():
    print("Cloning DeepPCB...")
    r = subprocess.run(["git", "clone", "--depth", "1",
                         "https://github.com/tangsanli5201/DeepPCB", str(DEEPPCB_DIR)],
                        capture_output=True, text=True)
    print((r.stdout or "")[-400:]); print((r.stderr or "")[-400:])
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed - download the ZIP manually into {DEEPPCB_DIR}")
else:
    print("DeepPCB already present at", DEEPPCB_DIR)

PCBDATA = DEEPPCB_DIR / "PCBData"
assert PCBDATA.exists(), f"Expected {PCBDATA} - top-level contents: {[p.name for p in DEEPPCB_DIR.iterdir()]}"
deeppcb_images = sorted(p for p in PCBDATA.rglob("*")
                         if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
print(f"DeepPCB bare-board images found: {len(deeppcb_images)}")

bare_pool = [(p, "pku") for p in pku_images] + [(p, "deeppcb") for p in deeppcb_images]
print(f"\nTotal bare_pcb pool: {len(bare_pool)}  (PKU={len(pku_images)}, DeepPCB={len(deeppcb_images)})")

## 4. Download the `pcba` pool (Roboflow)

Dataset: [`pcbdataset-abppt/pcb-components-rkizn`](https://universe.roboflow.com/pcbdataset-abppt/pcb-components-rkizn) —
populated-board component-detection images. We only need the images (bounding boxes discarded), so
we download in `yolov8` export format and glob every image regardless of split or label.

Requires a Roboflow account API key exported as `ROBOFLOW_API_KEY` (free tier is enough — this is a
public Universe project). Get one at https://app.roboflow.com/settings/api.

In [ ]:
from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY")
assert api_key, "Set ROBOFLOW_API_KEY in your environment before running this cell."

rf = Roboflow(api_key=api_key)
rf_project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

version_id = ROBOFLOW_VERSION
if version_id is None:
    versions = sorted(int(v.version) for v in rf_project.versions())
    assert versions, f"No published versions found for {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}"
    version_id = versions[-1]
print(f"Downloading {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} version {version_id} ...")

if not ROBOFLOW_DIR.exists():
    dataset = rf_project.version(version_id).download("yolov8", location=str(ROBOFLOW_DIR))
else:
    print(f"{ROBOFLOW_DIR} already present - delete it to re-download.")

pcba_images = sorted(p for p in ROBOFLOW_DIR.rglob("*")
                      if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
                      and "labels" not in p.parts)
pcba_pool = [(p, "roboflow_pcba") for p in pcba_images]
print(f"PCBA pool: {len(pcba_pool)} images")

## 5. Build the classification dataset

Stratified 70/15/15 split (seed-locked), optionally class-balanced by undersampling the majority
class so accuracy isn't inflated by imbalance. Images are hard-linked (falls back to copy) into
`pcb_vs_pcba_dataset/{train,val,test}/{bare_pcb,pcba}/` — an ImageFolder-compatible layout that both
Ultralytics `-cls` models and torchvision `ImageFolder` can consume directly.

In [ ]:
import random as _random
import shutil

_random.seed(SEED)

def stratified_split(items, ratios):
    items = items[:]
    _random.shuffle(items)
    n = len(items)
    n_train = int(n * ratios["train"])
    n_val   = int(n * ratios["val"])
    return {
        "train": items[:n_train],
        "val":   items[n_train:n_train + n_val],
        "test":  items[n_train + n_val:],
    }

bare_pool_bal, pcba_pool_bal = bare_pool, pcba_pool
if BALANCE_CLASSES:
    n_min = min(len(bare_pool), len(pcba_pool))
    bare_pool_bal = _random.sample(bare_pool, n_min)
    pcba_pool_bal = _random.sample(pcba_pool, n_min)
    print(f"Balanced both classes to {n_min} images each "
          f"(bare_pcb had {len(bare_pool)}, pcba had {len(pcba_pool)})")

splits = {
    "bare_pcb": stratified_split(bare_pool_bal, SPLIT_RATIOS),
    "pcba":     stratified_split(pcba_pool_bal, SPLIT_RATIOS),
}

if CLS_DATASET_DIR.exists():
    print(f"{CLS_DATASET_DIR} already exists - delete it to rebuild from scratch.")
else:
    manifest = []
    for cls_name, split_dict in splits.items():
        for split_name, items in split_dict.items():
            dst_dir = CLS_DATASET_DIR / split_name / cls_name
            dst_dir.mkdir(parents=True, exist_ok=True)
            for i, (src_path, source_tag) in enumerate(items):
                dst_path = dst_dir / f"{source_tag}_{src_path.stem}_{i}{src_path.suffix.lower()}"
                try:
                    os.link(src_path, dst_path)
                except OSError:
                    shutil.copy2(src_path, dst_path)
                manifest.append({"split": split_name, "class": cls_name,
                                  "source": source_tag, "path": str(dst_path)})
    import pandas as pd
    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(RESULTS_DIR / f"{EXP_NAME}_split_manifest.csv", index=False)
    print(manifest_df.groupby(["split", "class"]).size().unstack(fill_value=0))

In [ ]:
import pandas as pd

dist_rows = []
for split_name in ["train", "val", "test"]:
    for cls_name in CLASS_NAMES:
        d = CLS_DATASET_DIR / split_name / cls_name
        dist_rows.append({"split": split_name, "class": cls_name,
                           "n": len(list(d.glob("*"))) if d.exists() else 0})
dist_df = pd.DataFrame(dist_rows)
print(dist_df.pivot(index="class", columns="split", values="n"))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
pivot = dist_df.pivot(index="split", columns="class", values="n")
pivot.plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_title("bare_pcb vs pcba - split distribution"); ax.set_ylabel("images")
plt.tight_layout()
dist_plot_path = str(RESULTS_DIR / f"{EXP_NAME}_class_distribution.png")
plt.savefig(dist_plot_path, dpi=150)
plt.show()

## 6. W&B run

One run for the whole 3-model comparison (mirrors the `pcb-defect-detection` project convention from
Runs 1-8, in a sibling project so detection and classification experiments don't mix).

In [ ]:
import wandb

# wandb.login() reads WANDB_API_KEY from the environment / netrc - never hardcode the key.
wandb.login()

run = wandb.init(
    project="pcb-bare-vs-pcba-classification",
    name=EXP_NAME,
    tags=["classification", "bare-vs-pcba", "yolo11-cls", "efficientnet-b0", "mobilenetv3-small"],
    notes=(
        "3-model comparative benchmark: YOLOv11n-cls vs EfficientNet-B0 vs MobileNetV3-Small, "
        "bare_pcb (PKU-PCB + DeepPCB) vs pcba (Roboflow pcb-components-rkizn)."
    ),
    config=dict(
        seed=SEED, img_size=IMG_SIZE, batch=BATCH, epochs=EPOCHS, patience=PATIENCE,
        balance_classes=BALANCE_CLASSES, split_ratios=SPLIT_RATIOS,
        bare_pcb_sources=["PKU-PCB (Ironbrotherstyle/PCB-DATASET)", "DeepPCB (tangsanli5201/DeepPCB)"],
        pcba_source=f"roboflow:{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}",
    ),
    save_code=True,
)
wandb.log({"dataset/class_distribution": wandb.Table(dataframe=dist_df)})
wandb.log({"dataset/class_distribution_plot": wandb.Image(dist_plot_path)})
print("Run URL:", run.url)

## 7. Shared evaluation utilities

Used identically for every model so the comparison is apples-to-apples: test-set accuracy /
precision / recall / F1 / confusion matrix, parameter count, on-disk size, and CPU+GPU per-image
latency (batch=1, averaged over `N_LATENCY_RUNS` after a short warmup).

In [ ]:
import time
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

N_LATENCY_RUNS = 100
N_LATENCY_WARMUP = 10

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(state_dict_path):
    return Path(state_dict_path).stat().st_size / (1024 ** 2)

def classification_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return dict(accuracy=acc, precision=p, recall=r, f1=f1, confusion_matrix=cm)

def benchmark_latency_torch(model, img_size, device):
    model.eval().to(device)
    x = torch.randn(1, 3, img_size, img_size, device=device)
    with torch.no_grad():
        for _ in range(N_LATENCY_WARMUP):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(N_LATENCY_RUNS):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()
    ms_per_image = (t1 - t0) / N_LATENCY_RUNS * 1000
    return ms_per_image

def plot_confusion(cm, title, save_path):
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.show()
    return save_path

results_table = []   # one row per model, filled in as each model finishes

## 8. torchvision training/eval loop (shared by EfficientNet-B0 and MobileNetV3-Small)

Standard transfer-learning recipe: ImageNet-pretrained backbone, replaced classifier head, AdamW +
cosine schedule, early stopping on val accuracy, mixed precision on GPU.

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def make_loaders(batch_size):
    train_ds = datasets.ImageFolder(CLS_DATASET_DIR / "train", transform=train_tfms)
    val_ds   = datasets.ImageFolder(CLS_DATASET_DIR / "val",   transform=eval_tfms)
    test_ds  = datasets.ImageFolder(CLS_DATASET_DIR / "test",  transform=eval_tfms)
    assert train_ds.classes == CLASS_NAMES, f"ImageFolder class order {train_ds.classes} != {CLASS_NAMES}"
    kw = dict(batch_size=batch_size, num_workers=4, pin_memory=torch.cuda.is_available())
    return (DataLoader(train_ds, shuffle=True, **kw),
            DataLoader(val_ds, shuffle=False, **kw),
            DataLoader(test_ds, shuffle=False, **kw))

def train_torchvision_model(model, model_tag, device):
    train_loader, val_loader, test_loader = make_loaders(BATCH)
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    criterion = torch.nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    best_val_acc, best_state, epochs_no_improve = 0.0, None, 0
    for epoch in range(EPOCHS):
        model.train()
        run_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                out = model(xb)
                loss = criterion(out, yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            run_loss += loss.item() * xb.size(0)
        sched.step()
        train_loss = run_loss / len(train_loader.dataset)

        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                pred = model(xb).argmax(1).cpu().numpy()
                y_pred.extend(pred); y_true.extend(yb.numpy())
        val_acc = accuracy_score(y_true, y_pred)

        wandb.log({f"{model_tag}/train_loss": train_loss, f"{model_tag}/val_acc": val_acc,
                    f"{model_tag}/epoch": epoch, f"{model_tag}/lr": sched.get_last_lr()[0]})
        print(f"[{model_tag}] epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc, best_state, epochs_no_improve = val_acc, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"[{model_tag}] early stopping at epoch {epoch+1} (best val_acc={best_val_acc:.4f})")
                break

    model.load_state_dict(best_state)
    weights_path = RUNS_DIR / f"{model_tag}_best.pt"
    torch.save(best_state, weights_path)

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            pred = model(xb).argmax(1).cpu().numpy()
            y_pred.extend(pred); y_true.extend(yb.numpy())
    metrics = classification_metrics(y_true, y_pred)
    return model, metrics, weights_path

## 9. Model A — YOLOv11n-cls (Ultralytics)

In [ ]:
from ultralytics import YOLO

def train_yolo_cls(weights_name, model_tag):
    model = YOLO(weights_name)
    train_results = model.train(
        data=str(CLS_DATASET_DIR),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        seed=SEED,
        patience=PATIENCE,
        project=str(RUNS_DIR),
        name=model_tag,
        exist_ok=True,
    )
    best_weights = Path(model.trainer.best)
    test_model = YOLO(str(best_weights))
    val_results = test_model.val(data=str(CLS_DATASET_DIR), split="test", imgsz=IMG_SIZE, device=DEVICE)

    # Ultralytics classification val gives top1/top5 - derive full precision/recall/F1/confusion
    # matrix ourselves from raw predictions so the metric set matches the torchvision models exactly.
    test_dir = CLS_DATASET_DIR / "test"
    y_true, y_pred = [], []
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        for img_path in (test_dir / cls_name).glob("*"):
            pred = test_model.predict(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
            y_true.append(cls_idx); y_pred.append(int(pred.probs.top1))
    metrics = classification_metrics(y_true, y_pred)
    metrics["top1_ultralytics"] = float(val_results.top1)
    return test_model, metrics, best_weights

yolo_n_model, yolo_n_metrics, yolo_n_weights = train_yolo_cls("yolo11n-cls.pt", "yolov11n_cls")
print(yolo_n_metrics)

In [ ]:
cm_path = plot_confusion(yolo_n_metrics["confusion_matrix"], "YOLOv11n-cls - Test Confusion Matrix",
                          RESULTS_DIR / f"{EXP_NAME}_yolov11n_cls_confusion.png")

yolo_n_pt = torch.load(yolo_n_weights, map_location="cpu")
n_params_yolo_n = sum(p.numel() for p in yolo_n_model.model.parameters())
size_mb_yolo_n = model_size_mb(yolo_n_weights)

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lat_yolo_n_gpu = benchmark_latency_torch(yolo_n_model.model, IMG_SIZE, _device) if torch.cuda.is_available() else None
lat_yolo_n_cpu = benchmark_latency_torch(yolo_n_model.model, IMG_SIZE, torch.device("cpu"))

results_table.append(dict(
    model="YOLOv11n-cls", params_m=n_params_yolo_n / 1e6, size_mb=size_mb_yolo_n,
    accuracy=yolo_n_metrics["accuracy"], precision=yolo_n_metrics["precision"],
    recall=yolo_n_metrics["recall"], f1=yolo_n_metrics["f1"],
    latency_cpu_ms=lat_yolo_n_cpu, latency_gpu_ms=lat_yolo_n_gpu,
))
wandb.log({"yolov11n_cls/test_accuracy": yolo_n_metrics["accuracy"],
           "yolov11n_cls/test_f1": yolo_n_metrics["f1"],
           "yolov11n_cls/confusion_matrix": wandb.Image(str(cm_path))})
print(results_table[-1])

### 9b. (Optional) YOLOv11s-cls

Only runs if `INCLUDE_YOLO_S_CLS = True` in the config cell — kept optional since the brief lists
YOLOv11n/s-cls as one interchangeable row, not two separate required models.

In [ ]:
if INCLUDE_YOLO_S_CLS:
    yolo_s_model, yolo_s_metrics, yolo_s_weights = train_yolo_cls("yolo11s-cls.pt", "yolov11s_cls")
    cm_path_s = plot_confusion(yolo_s_metrics["confusion_matrix"], "YOLOv11s-cls - Test Confusion Matrix",
                                RESULTS_DIR / f"{EXP_NAME}_yolov11s_cls_confusion.png")
    n_params_yolo_s = sum(p.numel() for p in yolo_s_model.model.parameters())
    size_mb_yolo_s = model_size_mb(yolo_s_weights)
    lat_yolo_s_gpu = benchmark_latency_torch(yolo_s_model.model, IMG_SIZE, _device) if torch.cuda.is_available() else None
    lat_yolo_s_cpu = benchmark_latency_torch(yolo_s_model.model, IMG_SIZE, torch.device("cpu"))
    results_table.append(dict(
        model="YOLOv11s-cls", params_m=n_params_yolo_s / 1e6, size_mb=size_mb_yolo_s,
        accuracy=yolo_s_metrics["accuracy"], precision=yolo_s_metrics["precision"],
        recall=yolo_s_metrics["recall"], f1=yolo_s_metrics["f1"],
        latency_cpu_ms=lat_yolo_s_cpu, latency_gpu_ms=lat_yolo_s_gpu,
    ))
    wandb.log({"yolov11s_cls/test_accuracy": yolo_s_metrics["accuracy"],
               "yolov11s_cls/test_f1": yolo_s_metrics["f1"],
               "yolov11s_cls/confusion_matrix": wandb.Image(str(cm_path_s))})
    print(results_table[-1])
else:
    print("INCLUDE_YOLO_S_CLS is False - skipping.")

## 10. Model B — EfficientNet-B0 (torchvision, ImageNet-pretrained)

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
in_feats = effnet.classifier[1].in_features
effnet.classifier[1] = torch.nn.Linear(in_feats, len(CLASS_NAMES))

effnet, effnet_metrics, effnet_weights = train_torchvision_model(effnet, "efficientnet_b0", _device)
print(effnet_metrics)

In [ ]:
cm_path_eff = plot_confusion(effnet_metrics["confusion_matrix"], "EfficientNet-B0 - Test Confusion Matrix",
                              RESULTS_DIR / f"{EXP_NAME}_efficientnet_b0_confusion.png")

n_params_eff = count_params(effnet)
size_mb_eff = model_size_mb(effnet_weights)
lat_eff_gpu = benchmark_latency_torch(effnet, IMG_SIZE, _device) if torch.cuda.is_available() else None
lat_eff_cpu = benchmark_latency_torch(effnet, IMG_SIZE, torch.device("cpu"))

results_table.append(dict(
    model="EfficientNet-B0", params_m=n_params_eff / 1e6, size_mb=size_mb_eff,
    accuracy=effnet_metrics["accuracy"], precision=effnet_metrics["precision"],
    recall=effnet_metrics["recall"], f1=effnet_metrics["f1"],
    latency_cpu_ms=lat_eff_cpu, latency_gpu_ms=lat_eff_gpu,
))
wandb.log({"efficientnet_b0/test_accuracy": effnet_metrics["accuracy"],
           "efficientnet_b0/test_f1": effnet_metrics["f1"],
           "efficientnet_b0/confusion_matrix": wandb.Image(str(cm_path_eff))})
print(results_table[-1])

## 11. Model C — MobileNetV3-Small (torchvision, ImageNet-pretrained)

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

mnet = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
in_feats = mnet.classifier[3].in_features
mnet.classifier[3] = torch.nn.Linear(in_feats, len(CLASS_NAMES))

mnet, mnet_metrics, mnet_weights = train_torchvision_model(mnet, "mobilenet_v3_small", _device)
print(mnet_metrics)

In [ ]:
cm_path_mnet = plot_confusion(mnet_metrics["confusion_matrix"], "MobileNetV3-Small - Test Confusion Matrix",
                               RESULTS_DIR / f"{EXP_NAME}_mobilenet_v3_small_confusion.png")

n_params_mnet = count_params(mnet)
size_mb_mnet = model_size_mb(mnet_weights)
lat_mnet_gpu = benchmark_latency_torch(mnet, IMG_SIZE, _device) if torch.cuda.is_available() else None
lat_mnet_cpu = benchmark_latency_torch(mnet, IMG_SIZE, torch.device("cpu"))

results_table.append(dict(
    model="MobileNetV3-Small", params_m=n_params_mnet / 1e6, size_mb=size_mb_mnet,
    accuracy=mnet_metrics["accuracy"], precision=mnet_metrics["precision"],
    recall=mnet_metrics["recall"], f1=mnet_metrics["f1"],
    latency_cpu_ms=lat_mnet_cpu, latency_gpu_ms=lat_mnet_gpu,
))
wandb.log({"mobilenet_v3_small/test_accuracy": mnet_metrics["accuracy"],
           "mobilenet_v3_small/test_f1": mnet_metrics["f1"],
           "mobilenet_v3_small/confusion_matrix": wandb.Image(str(cm_path_mnet))})
print(results_table[-1])

## 12. Comparative results

In [ ]:
results_df = pd.DataFrame(results_table).set_index("model")
results_df = results_df.round(4)
print(results_df.to_string())

results_csv = RESULTS_DIR / f"{EXP_NAME}_comparison.csv"
results_df.to_csv(results_csv)
wandb.log({"comparison/results_table": wandb.Table(dataframe=results_df.reset_index())})
results_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].bar(results_df.index, results_df["accuracy"], color="#4C72B0")
axes[0, 0].set_title("Test Accuracy"); axes[0, 0].set_ylim(0, 1); axes[0, 0].tick_params(axis="x", rotation=20)

axes[0, 1].bar(results_df.index, results_df["f1"], color="#55A868")
axes[0, 1].set_title("Test F1"); axes[0, 1].set_ylim(0, 1); axes[0, 1].tick_params(axis="x", rotation=20)

axes[1, 0].bar(results_df.index, results_df["params_m"], color="#C44E52")
axes[1, 0].set_title("Parameters (M)"); axes[1, 0].tick_params(axis="x", rotation=20)

axes[1, 1].bar(results_df.index, results_df["latency_cpu_ms"], color="#8172B2", label="CPU")
if results_df["latency_gpu_ms"].notna().any():
    axes[1, 1].bar(results_df.index, results_df["latency_gpu_ms"], color="#CCB974", alpha=0.7, label="GPU")
    axes[1, 1].legend()
axes[1, 1].set_title("Latency per image (ms, batch=1)"); axes[1, 1].tick_params(axis="x", rotation=20)

plt.suptitle("Bare-PCB vs PCBA - 3-Model Comparison", fontweight="bold")
plt.tight_layout()
comparison_plot_path = str(RESULTS_DIR / f"{EXP_NAME}_comparison_plot.png")
plt.savefig(comparison_plot_path, dpi=150)
wandb.log({"comparison/plot": wandb.Image(comparison_plot_path)})
plt.show()

## 13. Log model artifacts + close out W&B run

In [ ]:
for tag, path in [("yolov11n_cls", yolo_n_weights), ("efficientnet_b0", effnet_weights),
                   ("mobilenet_v3_small", mnet_weights)]:
    artifact = wandb.Artifact(
        name=f"{EXP_NAME}_{tag}", type="model",
        description=f"{tag} - bare_pcb vs pcba classifier",
        metadata=results_df.loc[[c for c in results_df.index if tag.split('_')[0] in c.lower().replace('-', '_')]].to_dict()
                  if any(tag.split('_')[0] in c.lower().replace('-', '_') for c in results_df.index) else {},
    )
    artifact.add_file(str(path))
    wandb.log_artifact(artifact)

wandb.finish()
print("Done. Results saved to:", results_csv)

## 14. Notes / how to read the results

- **Accuracy/F1** are on the held-out **test** split (never seen during training or model selection);
  the model checkpoint used is the best-val-accuracy epoch (early stopping, patience=`PATIENCE`).
- **Params/size/latency** are measured on the *same* trained weights used for the accuracy numbers —
  this is an accuracy-vs-cost comparison, not two separate benchmarks.
- If framing this as a **pre-filter stage** ahead of the defect-detection pipeline (Runs 1-8): the
  relevant trade-off is MobileNetV3-Small/YOLOv11n-cls's latency/size advantage vs. any accuracy gap
  to EfficientNet-B0 — a few points of accuracy matters less than throughput if this gate runs on
  every incoming image before the (much heavier) detection stage.
- Class balance: `BALANCE_CLASSES=True` undersamples the majority class 1:1 so accuracy isn't
  inflated by imbalance; check `results/{EXP_NAME}_split_manifest.csv` for the exact per-source counts
  if you want to re-run with `BALANCE_CLASSES=False` instead.
- Known caveat: the `pcba` pool comes from a single Roboflow source (one imaging setup/scale), while
  `bare_pcb` blends two sources (PKU + DeepPCB, different modalities). A model could partly learn
  "which imaging pipeline" rather than "is this populated" - worth a held-out sanity check with a
  second PCBA source before trusting this as a generalizable pre-filter.